In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [6]:
chain = prompt | llm | output_parser

## NER Model

In [7]:
import spacy
from span_marker import SpanMarkerModel

In [8]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [9]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [10]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [11]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

## Pipeline Entry Point

In [12]:
# Temporary. Use given article data set
full_df = pd.read_csv("./sample_data/Articles_Nov_2020_March_2023.csv")

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(10)
# raw_df = full_df
len(raw_df)

10

In [13]:
# Uncomment later 
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [14]:
# raw_df = pd.read_csv(sample_data_dir)

In [15]:
raw_df.head(3)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Article,Mayor Wu unveils Boston Common master plan,Mayor Wu unveils Boston Common master plan,Frankie Rowley,NaN,Local News,NaN,/local-news/2022/10/12/mayor-wu-unveils-boston...,Wed Oct 12 15:23:22 EDT 2022,TRUE,Mayor Michelle Wu announced the Boston Common ...
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Article,Buy a rural hospital for $100? Investors pick ...,Buy a rural hospital for $100? Investors pick ...,Blake Farmer,NaN,National News,NaN,/national-news/2022/08/16/buy-a-rural-hospital...,Tue Aug 16 05:00:00 EDT 2022,TRUE,"ERIN, Tenn. — Kyle Kopec gets a kick out of le..."
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Article,Is it time for a reality check on rapid COVID ...,Is it time for a reality check on rapid COVID ...,Sydney Lupkin,NaN,National News,NaN,/national-news/2023/01/20/is-it-time-for-a-rea...,Thu Jan 19 12:02:00 EST 2023,TRUE,As the COVID-19 pandemic enters its fourth yea...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [16]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [17]:
# For Testing Purposes Only
# df = df[:20]

In [18]:
df["llama_prediction"] = None # Add the llama_prediction

Remove Duplicates (if any)

In [19]:
duplicates = df.duplicated(subset=['hl1'])

In [20]:
print(duplicates.value_counts())

False    10
Name: count, dtype: int64


In [21]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header -> Regex Cleaner

In [22]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

  0%|          | 0/10 [00:00<?, ?it/s]C:\Users\axel0\AppData\Local\Temp\ipykernel_24728\1969457438.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
100%|██████████| 10/10 [00:00<00:00, 9998.34it/s]


In [23]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 10/10 [00:00<?, ?it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary

In [24]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    known_locs_path = "./geodata/known_locs.json"
    # Load known locations of the form (location: [lat, long])
    with open(known_locs_path, 'r') as file:
        known_locs_dict = json.load(file) 
    
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_locs_dict.keys():
        # if (key in lowercase_header):
        if (location.lower() in lowercase_header):
            return [location, known_locs_dict[location]]   
    return None

In [25]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 10/10 [00:00<00:00, 564.70it/s]


In [26]:
df["Explicit_Pass"].value_counts().head(10)

Explicit_Pass
[Boston Common, {'lat': 42.3550897, 'lon': -71.0657256}]          1
[BU, {'lat': 42.3504997, 'lon': -71.1053991}]                     1
[Revere, {'lat': 42.4084302, 'lon': -71.0119948}]                 1
[Boston, {'lat': 42.3600825, 'lon': -71.0588801}]                 1
[Boston Public Radio, {'lat': 42.3600825, 'lon': -71.0588801}]    1
Name: count, dtype: int64

In [27]:
# Extract the coordinates from the explicit pass
def extract_coordinates(entities):
    if entities is None:
        return None
    
    coordinates = entities[1]
    latitude = coordinates['lat']
    longitude = coordinates['lon']
    formatted_coordinates = [str(longitude), str(latitude)]
    return formatted_coordinates

df['Explicit_Pass_Coordinates'] = df["Explicit_Pass"].progress_apply(extract_coordinates)

100%|██████████| 10/10 [00:00<?, ?it/s]


In [28]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]"
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]"
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,None,None,None


### NER Code First Pass

In [29]:
# Run NER on the body of the article and return all entities
def predict_NER_def(text):
    try:
        if (text == None or text == ""):
            return None
        
        # Extract entities from the text
        entities = []
        for entity in nlp(text).ents:
            entities.append((entity.text, entity.label_))

        return entities
    except Exception as error:
        print(error) # Double prints error?
        return None

In [30]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return None

In [31]:
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

  0%|          | 0/10 [00:00<?, ?it/s]

Has location from title: Mayor Wu unveils Boston Common master plan
Has location from title: Buy rural hospital for $100 Investors pick up struggling institutions for pennies


 70%|███████   | 7/10 [14:32<07:18, 146.24s/it]

Has location from title: Nearly Half Of Revere Public School Staff Receive Vaccine Shots In One Day
Has location from title: Supreme Court Decision To Consider Boston Marathon Bombing Case Could Have Wide Ranging Ripple Effects On High Profile Cases
Has location from title: Boston Public Radio Full Show 22 21


100%|██████████| 10/10 [22:34<00:00, 135.48s/it]


Filter based on superb specific places such as 'FAC'.

In [32]:
# Filter out the entities that are not locations and sort them by priorities
def filter_locations(entities):

    if (entities == None):
        return None
    
    locations = []
    for entity in entities:
        # Check if entity is of the format (location, label)
        if (len(entity) >= 2): 
            location, label = entity
            
            # If the entity is a location, add it to the list
            if (("GPE" in label 
                    and "Boston" not in location 
                    and "Massachusetts" not in location) # Geopolitical Entity but not general like Boston / MA
                 or ("ORG" in label) # Organization
                 or ("FAC" in label) # Facility
                 or ("LOC" in label) # Location
                ):
                locations.append((location, label.strip()))
    
    # Sort the locations by priority
    priority_order = {'FAC': 1, 'ORG': 2, 'LOC': 3, 'GPE': 4}
    sorted_locations = sorted(locations, key=lambda entity: priority_order[entity[1]])
    
    return sorted_locations

In [33]:
df['NER_Pass_Sorted'] = df['NER_Pass'].progress_apply(filter_locations)

100%|██████████| 10/10 [00:00<?, ?it/s]


In [34]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,None,None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti..."


Functions to manage caches

In [ ]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

In [ ]:
# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file)

We can further filter the list now that it's sorted to pick out the most specific location that isn't an unwanted one like "Boston."

In [ ]:
# TODO: Populate the unwanted entities cache
unwanted_entities_path = "./geodata/known_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [ ]:
## TODO: DELETE AFTER FIRST RUN
unwanted_entities = {
    'FAC': ['Boston'],
    'ORG': ['New York Times'],
    'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
    'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
}

save_cache_to_file(unwanted_entities, unwanted_entities_path)

In [ ]:
# Return the first most specific entity that is not in the unwanted list
def getSpecificEntity(entities, valid_labels):
    if (entities == None):
        return None
    
    # Look through sorted entities, return first entity that is not unwanted
    for (location, label) in entities:
        if label in valid_labels:
            if location not in unwanted_entities[label]:
                return (location, label)
        else: 
            break
        
    # If no wanted entities found, return the first entity if it's valid
    location, label = entities[0]
    if label in valid_labels:
        return (location, label)
    else:
        return None

Then we get the coordinates through the first pass

In [ ]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

Get the coordinates only if it's a new location

In [ ]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [35]:
# Get the coordinates of the location
def getCoordinates(entities, valid_labels): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (entities == None or len(entities) == 0): return None  
    
    # Get first most specific wanted entity
    entity = getSpecificEntity(entities, valid_labels)
    if (entity == None): return None
    
    location = entity[0]
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [36]:
df['NER_Pass_Coordinates'] = df['NER_Pass_Sorted'].progress_apply(getCoordinates, valid_labels=['FAC'])

100%|██████████| 10/10 [00:00<00:00, 26.00it/s]


In [37]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted,NER_Pass_Coordinates
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None,None
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None,None
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,None,None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti...",None


### Llama Prediction

In [38]:
# Run the LLM model on the articles that haven't been tagged with a location yet
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass_Coordinates'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            return chain.invoke({"headline": article['hl1'], "body": article['body']})
    except Exception as error:
        print(error)
        return None

In [39]:
df['llama_prediction'] = df.progress_apply(predict_llama, axis=1)

  0%|          | 0/10 [00:00<?, ?it/s]

Has location from title or NER: Mayor Wu unveils Boston Common master plan
Has location from title or NER: Buy rural hospital for $100 Investors pick up struggling institutions for pennies
  Based on the article provided, I would guess that the location being referred to is likely Boston, Massachusetts. The article mentions the "Food and Drug Administration" and "the National Institutes of Health," which are both located in Boston. Additionally, the article quotes Dr. Robin Colgrove, who is a professor at Harvard Medical School and chair of the Diagnostics Committee of the Infectious Diseases Society of America, which is also located in Boston.
The specific location within Boston that is mentioned in the article is the "Luminostics Inc. Clip COVID Rapid Antigen Test," which is one of the tests that has been rendered less reliable in the face of new variants.
The involved specific locations or organizations explicitly found within the article that influenced my decision are:
1. Boston, 


llama_print_timings:        load time =   94771.00 ms
llama_print_timings:      sample time =      96.68 ms /   256 runs   (    0.38 ms per token,  2647.99 tokens per second)
llama_print_timings: prompt eval time =  166740.18 ms /  1769 tokens (   94.26 ms per token,    10.61 tokens per second)
llama_print_timings:        eval time =   53484.39 ms /   255 runs   (  209.74 ms per token,     4.77 tokens per second)
llama_print_timings:       total time =  221237.87 ms /  2024 tokens
 40%|████      | 4/10 [03:41<05:31, 55.33s/it]

Has location from title or NER: Plane of migrants arrives on Martha Vineyard unannounced island mobilizes with aid
Has location from title or NER: Who were the four chaplains of Four Chaplains Day


Llama.generate: prefix-match hit


  Here is my response to your request:
1. Y - The article is talking about a region of Boston as the airlines mentioned in the article are all based in Boston.
2. The specific location within Boston that I believe the article is referring to is the Boston Logan International Airport. This is based on the fact that all of the airlines mentioned in the article are headquartered in Boston and Logan Airport is the primary airport serving the city.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision include:
* Boston Logan International Airport (as mentioned above)
* The Centers for Disease Control and Prevention (CDC)
* The Biden administration
* Federal Judge Kathryn Kimball Mizelle

Based on the information provided in the article, it appears that the news of federal judge ruling against the Biden administration's mandatory mask mandate for travelers onboard airplanes and other forms of public transportation is influencing t


llama_print_timings:        load time =   94771.00 ms
llama_print_timings:      sample time =      94.52 ms /   256 runs   (    0.37 ms per token,  2708.45 tokens per second)
llama_print_timings: prompt eval time =   64178.35 ms /   732 tokens (   87.68 ms per token,    11.41 tokens per second)
llama_print_timings:        eval time =   51196.84 ms /   255 runs   (  200.77 ms per token,     4.98 tokens per second)
llama_print_timings:       total time =  116233.23 ms /   987 tokens
100%|██████████| 10/10 [05:37<00:00, 33.77s/it]

Has location from title or NER: Nearly Half Of Revere Public School Staff Receive Vaccine Shots In One Day
Has location from title or NER: Supreme Court Decision To Consider Boston Marathon Bombing Case Could Have Wide Ranging Ripple Effects On High Profile Cases
Has location from title or NER: Boston Public Radio Full Show 22 21
Requested tokens (3140) exceed context window of 2048


In [40]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted,NER_Pass_Coordinates
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None,None
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None,None
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,"Based on the article provided, I would guess...",None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti...",None


We then apply NER on the llama outputs

In [41]:
df['NER_Prediction'] = df['llama_prediction'].progress_apply(predict_NER_def)

100%|██████████| 10/10 [01:46<00:00, 10.64s/it]


In [42]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted,NER_Pass_Coordinates,NER_Prediction
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None,None,None
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None,None,None
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,"Based on the article provided, I would guess...",None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti...",None,"[(Boston, GPE), (Massachusetts, GPE), (the ""Fo..."


Sort the NER Predictions

In [43]:
df['NER_Pred_Sorted'] = df['NER_Prediction'].progress_apply(filter_locations)

100%|██████████| 10/10 [00:00<00:00, 666.24it/s]


Get the coordinates of the most specific location, which in this case we'll allow both FACilities and ORGanizations as it's the final pass.

In [44]:
df['NER_Pred_Coordinates'] = df['NER_Pred_Sorted'].progress_apply(getCoordinates, valid_labels=['FAC', 'ORG'])

100%|██████████| 10/10 [00:00<00:00, 16.70it/s]


In [45]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted,NER_Pass_Coordinates,NER_Prediction,NER_Pred_Sorted,NER_Pred_Coordinates
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None,None,None,None,None
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None,None,None,None,None
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,"Based on the article provided, I would guess...",None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti...",None,"[(Boston, GPE), (Massachusetts, GPE), (the ""Fo...","[(the ""Food and Drug Administration"", ORG), (t...","[-71.3824374, 42.4072107]"
10261,00000183-4068-df11-ad9b-586bf5b20001,Plane of migrants arrives on Martha Vineyard u...,Less than 24 hours after 50 migrants from Vene...,None,None,None,"[(Less than 24 hours, TIME), (50, CARDINAL), (...","[(St. Andrew Episcopal Church, FAC), (St. Andr...","[-71.3824374, 42.4072107]",None,None,None
7493,0000017e-d00b-d936-a37f-f2eb66370001,Who were the four chaplains of Four Chaplains Day,On February 3rd the U.S. military and countles...,None,None,None,"[(February 3rd, DATE), (U.S., GPE), (Four Chap...","[(the Navy Yard, FAC), (Boston University, ORG...","[-71.3824374, 42.4072107]",None,None,None
8538,00000180-3f8c-d3f4-a1dc-bfef61460001,Some major U.S. airlines are dropping mask man...,Following the news of federal judge in Florida...,Here is my response to your request:\n1. Y -...,None,None,"[(Florida, GPE), (Biden, PERSON), (Monday, DAT...","[(Delta Air Lines, ORG), (United Airlines, ORG...",None,"[(1, CARDINAL), (Boston, GPE), (Boston, GPE), ...","[(the Boston Logan International Airport, FAC)...","[-71.01078319999999, 42.3653985]"
2776,00000178-2883-db99-a17b-3ccb26260001,Nearly Half Of Revere Public School Staff Rece...,The race to get Massachusetts teachers vaccina...,None,"[Revere, {'lat': 42.4084302, 'lon': -71.0119948}]","[-71.0119948, 42.4084302]",None,None,None,None,None,None
2969,00000178-5f75-d00c-a1ff-dff57e400001,Supreme Court Decision To Consider Boston Mara...,The U.S. Supreme Court will review last summer...,None,"[Boston, {'lat': 42.3600825, 'lon': -71.0588801}]","[-71.0588801, 42.3600825]",None,None,None,None,None,None
2392,00000177-cb11-dbe7-ad77-ebdfaf420001,Boston Public Radio Full Show 22 21,Today on Boston Public Radio Brian McGrory wei...,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...","[-71.0588801, 42.3600825]",None,None,None,None,None,None
9515,00000181-e267-d804-a7ed-e26734d40001,She was already battling cancer. Then she had ...,RAPID CITY S.D. Jeni Rae Peters would make pro...,None,None,None,"[(RAPID CITY, GPE), (S.D., GPE), (Jeni Rae Pet...","[(KHN, ORG), (NPR, ORG), (KFF, ORG), (Kaiser F...",None,None,None,None


## Geocode locations

In [46]:
# Get the coordinates from the most specific pass for a given article
def extractCoordinates(article):
    coordinates = []

    if (article['Explicit_Pass_Coordinates'] != None): # From title
        coordinates = article['Explicit_Pass_Coordinates']
    elif(article['NER_Pass_Coordinates'] != None): # From NER Pass
        coordinates = article['NER_Pass_Coordinates']
    elif (article['NER_Pred_Coordinates'] != None): # From LLM Pass
        coordinates = article['NER_Pred_Coordinates']
    else: # Must be a very hard/bad article :-(
        return None 
    
    return coordinates

In [47]:
# TO Do: Save the geocodes to a json file. Should include location as key, coordinates, census tract, and conty as values
# If location in file, don't run geocode (both google maps and census tract). Just return the values
# Else run as expected

def save_geocodes(new_data, boston=True):
    # save new geocodes to json file 
    # one for boston locations 
    # one for locations outside of boston
    
    if boston:
        filename = "./entity_recognition/saved-geocodes.json"
        with open(filename, 'r+') as f:
            # load existing data 
            file_data = json.load(f)
            for name in new_data:
                file_data[name] = new_data[name]
            f.seek(0)
            # convert back to json
            json.dump(file_data, f, indent=4)
    """
    else:
        filename = "./entity_recognition/not-boston-saved-geocodes.json"
        with open(filename, 'r+') as f:
            # load existing data 
            file_data = json.load(f)
            for name in new_data:
                file_data[name] = new_data[name]
            f.seek(0)
            # convert back to json
            json.dump(file_data, f, indent=4)
    """

In [48]:
# Get the census tract of the location
def query_census_api(id, longitude, latitude):

    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + id)
        except KeyError:
            print("Location is outside of the United States: " + id)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [49]:
# Get the census tract of the location depending on which pass it was obtained from
def getTract(article):
    coordinates = extractCoordinates(article)
    if (coordinates == None):
        return None, None
    
    longitude, latitude = coordinates
    Tract, County = query_census_api(article['_id'], longitude, latitude)

    return Tract, County

In [50]:
# Save the tracts and counties into one column
df['Tracts_County'] = df.progress_apply(getTract, axis=1)
df[['Tracts', 'County']] = pd.DataFrame(df['Tracts_County'].tolist(), index=df.index)
df.drop(columns=['Tracts_County'], inplace=True)

100%|██████████| 10/10 [00:03<00:00,  3.02it/s]


In [51]:
df.head(3)

,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted,NER_Pass_Coordinates,NER_Prediction,NER_Pred_Sorted,NER_Pred_Coordinates,Tracts,County
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None,None,None,None,None,981700,025
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None,None,None,None,None,010103,025
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,"Based on the article provided, I would guess...",None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti...",None,"[(Boston, GPE), (Massachusetts, GPE), (the ""Fo...","[(the ""Food and Drug Administration"", ORG), (t...","[-71.3824374, 42.4072107]",365100,017


In [52]:
len(df)

10

In [53]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [54]:
print(len(df))
df

9


,_id,hl1,body,llama_prediction,Explicit_Pass,Explicit_Pass_Coordinates,NER_Pass,NER_Pass_Sorted,NER_Pass_Coordinates,NER_Prediction,NER_Pred_Sorted,NER_Pred_Coordinates,Tracts,County
10593,00000183-cd8f-d9b5-ab83-cfdf55ae0001,Mayor Wu unveils Boston Common master plan,Mayor Michelle Wu announced the Boston Common ...,None,"[Boston Common, {'lat': 42.3550897, 'lon': -71...","[-71.0657256, 42.3550897]",None,None,None,None,None,None,981700,025
9943,00000182-a619-d8f2-a1fa-ee9d78370001,Buy rural hospital for $100 Investors pick up ...,ERIN Tenn. Kyle Kopec gets kick out of leading...,None,"[BU, {'lat': 42.3504997, 'lon': -71.1053991}]","[-71.1053991, 42.3504997]",None,None,None,None,None,None,010103,025
11739,00000185-cb30-d12f-a1df-cbfe35cb0001,Is it time for reality check on rapid COVID tests,As the COVID 19 pandemic enters its fourth yea...,"Based on the article provided, I would guess...",None,None,"[(COVID 19, PRODUCT), (fourth year, DATE), (15...","[(Harvard Medical School, ORG), (the Diagnosti...",None,"[(Boston, GPE), (Massachusetts, GPE), (the ""Fo...","[(the ""Food and Drug Administration"", ORG), (t...","[-71.3824374, 42.4072107]",365100,017
10261,00000183-4068-df11-ad9b-586bf5b20001,Plane of migrants arrives on Martha Vineyard u...,Less than 24 hours after 50 migrants from Vene...,None,None,None,"[(Less than 24 hours, TIME), (50, CARDINAL), (...","[(St. Andrew Episcopal Church, FAC), (St. Andr...","[-71.3824374, 42.4072107]",None,None,None,365100,017
7493,0000017e-d00b-d936-a37f-f2eb66370001,Who were the four chaplains of Four Chaplains Day,On February 3rd the U.S. military and countles...,None,None,None,"[(February 3rd, DATE), (U.S., GPE), (Four Chap...","[(the Navy Yard, FAC), (Boston University, ORG...","[-71.3824374, 42.4072107]",None,None,None,365100,017
8538,00000180-3f8c-d3f4-a1dc-bfef61460001,Some major U.S. airlines are dropping mask man...,Following the news of federal judge in Florida...,Here is my response to your request:\n1. Y -...,None,None,"[(Florida, GPE), (Biden, PERSON), (Monday, DAT...","[(Delta Air Lines, ORG), (United Airlines, ORG...",None,"[(1, CARDINAL), (Boston, GPE), (Boston, GPE), ...","[(the Boston Logan International Airport, FAC)...","[-71.01078319999999, 42.3653985]",981300,025
2776,00000178-2883-db99-a17b-3ccb26260001,Nearly Half Of Revere Public School Staff Rece...,The race to get Massachusetts teachers vaccina...,None,"[Revere, {'lat': 42.4084302, 'lon': -71.0119948}]","[-71.0119948, 42.4084302]",None,None,None,None,None,None,170601,025
2969,00000178-5f75-d00c-a1ff-dff57e400001,Supreme Court Decision To Consider Boston Mara...,The U.S. Supreme Court will review last summer...,None,"[Boston, {'lat': 42.3600825, 'lon': -71.0588801}]","[-71.0588801, 42.3600825]",None,None,None,None,None,None,030302,025
2392,00000177-cb11-dbe7-ad77-ebdfaf420001,Boston Public Radio Full Show 22 21,Today on Boston Public Radio Brian McGrory wei...,None,"[Boston Public Radio, {'lat': 42.3600825, 'lon...","[-71.0588801, 42.3600825]",None,None,None,None,None,None,030302,025


## Topic Modeling

In [55]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [56]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [57]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [58]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

FileNotFoundError: [Errno 2] No such file or directory: './taxonomy_list/Content_Taxonomy.csv'

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns